**Name:** Ruikun Zhang  
**GitHub Username:** XOBECIUJ  
**USC ID:** 6347-9436-32

# HW2 - Combined Cycle Power Plant Regression
DSCI-552

In [ ]:
!pip install -q statsmodels

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

sns.set_style('whitegrid')

## (a) Load the data
Using Sheet1 of the Combined Cycle Power Plant dataset.

In [ ]:
df = pd.read_excel('../data/Folds5x2_pp.xlsx', sheet_name='Sheet1')
df = df.rename(columns={'AT': 'T', 'PE': 'EP'})
predictors = ['T', 'AP', 'RH', 'V']
df.head()

## (b) Exploring the data
### (i) Rows and columns

In [ ]:
print('rows:', df.shape[0], ' columns:', df.shape[1])
df.info()

There are 9568 rows, one per hourly reading taken at the power plant, and 5 columns: T (ambient temperature), AP (ambient pressure), RH (relative humidity) and V (exhaust vacuum) are the predictors, and EP (net hourly electrical energy output) is the response.

### (ii) Pairwise scatterplots

In [ ]:
sns.pairplot(df)
plt.show()

T and V both show a strong negative, fairly linear relationship with EP. AP has a weaker positive relationship with EP, and RH an even weaker positive one. T and V are also strongly correlated with each other, which will matter later when interpreting the multiple regression coefficients.

### (iii) Summary statistics

In [ ]:
summary = df.describe().T[['mean', '50%', 'min', 'max', '25%', '75%']].copy()
summary.columns = ['mean', 'median', 'min', 'max', 'Q1', 'Q3']
summary['range'] = summary['max'] - summary['min']
summary['IQR'] = summary['Q3'] - summary['Q1']
summary = summary[['mean', 'median', 'range', 'Q1', 'Q3', 'IQR']]
summary

## (c) Simple linear regression for each predictor

In [ ]:
for p in predictors:
    m = smf.ols(f'EP ~ {p}', data=df).fit()
    print(p, ' coef=%.4f' % m.params[p], ' p-value=%.3g' % m.pvalues[p], ' R2=%.4f' % m.rsquared)

All four predictors are statistically significant on their own (p-values essentially 0). T has by far the strongest univariate relationship with EP (R² = 0.90), followed by V (R² = 0.76), then AP (R² = 0.27) and RH (R² = 0.15).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, p in zip(axes.flatten(), predictors):
    ax.scatter(df[p], df['EP'], alpha=0.2, s=8)
    x_sorted = np.sort(df[p].values)
    m = smf.ols(f'EP ~ {p}', data=df).fit()
    ax.plot(x_sorted, m.predict(pd.DataFrame({p: x_sorted})), color='red')
    ax.set_xlabel(p)
    ax.set_ylabel('EP')
plt.tight_layout()
plt.show()

In [ ]:
# flag points with a large residual (more than 3 standard deviations) as possible outliers
for p in predictors:
    m = smf.ols(f'EP ~ {p}', data=df).fit()
    resid = m.resid
    outliers = df[np.abs(resid) > 3 * resid.std()]
    print(p, ' potential outliers:', len(outliers))

Each predictor has a small number (well under 1% of the data) of points with unusually large residuals. Given how few there are and that the dataset is a large, clean set of physical sensor readings, there is no strong reason to remove them before fitting the regression models below.

## (d) Multiple regression using all predictors

In [ ]:
mfull = smf.ols('EP ~ T + AP + RH + V', data=df).fit()
mfull.summary()

R² = 0.929, higher than any single predictor alone. All four p-values are far below 0.05, so we reject H0: beta_i = 0 for T, AP, RH and V — every predictor is statistically significant even after controlling for the other three.

## (e) Comparing simple vs. multiple regression coefficients

In [ ]:
simple_coef = {p: smf.ols(f'EP ~ {p}', data=df).fit().params[p] for p in predictors}
multi_coef = {p: mfull.params[p] for p in predictors}

plt.figure(figsize=(6, 6))
for p in predictors:
    plt.scatter(simple_coef[p], multi_coef[p])
    plt.annotate(p, (simple_coef[p], multi_coef[p]))
plt.axhline(0, color='gray', linewidth=0.5)
plt.axvline(0, color='gray', linewidth=0.5)
plt.xlabel('Simple regression coefficient')
plt.ylabel('Multiple regression coefficient')
plt.show()

AP's coefficient shrinks a lot (1.49 -> 0.06) and RH's coefficient even flips sign (0.46 -> -0.16) once the other predictors are in the model. This is a sign of correlation among the predictors (T and V in particular): once you control for the other variables, a predictor's coefficient no longer has the same simple interpretation it had on its own.

## (f) Nonlinear association with each predictor

In [ ]:
for p in predictors:
    m = smf.ols(f'EP ~ {p} + I({p}**2) + I({p}**3)', data=df).fit()
    p2 = m.pvalues[f'I({p} ** 2)']
    p3 = m.pvalues[f'I({p} ** 3)']
    print(p, ' quadratic p=%.3g' % p2, ' cubic p=%.3g' % p3)

T, AP and RH all have significant quadratic and cubic terms, so their relationship with EP is not purely linear. V is different: its quadratic term is not significant (p = 0.77), while its cubic term barely is (p = 0.01), so the evidence for nonlinearity in V is much weaker than for the other three predictors.

## (g) Interaction terms

In [ ]:
mint = smf.ols('EP ~ (T + AP + RH + V) ** 2', data=df).fit()
mint.pvalues

Most of the pairwise interaction terms are statistically significant (T:RH, T:V, AP:RH, AP:V all have p well below 0.05), which means the effect of one predictor on EP does depend on the level of another. T:AP and RH:V are the two interactions that are not significant.

## (h) Improving the model
Train on a random 70% of the data and test on the remaining 30%. Model A uses all four predictors with no interactions or nonlinear terms. Model B starts from all pairwise interactions and quadratic terms and removes insignificant terms one at a time using p-values, keeping a main effect in the model as long as an interaction involving it is still significant (this is the same hierarchy principle covered in lecture).

In [ ]:
train, test = train_test_split(df, test_size=0.3, random_state=1)

mA = smf.ols('EP ~ T + AP + RH + V', data=train).fit()
mseA_train = mean_squared_error(train['EP'], mA.predict(train))
mseA_test = mean_squared_error(test['EP'], mA.predict(test))
print('Model A train MSE=%.3f test MSE=%.3f' % (mseA_train, mseA_test))

In [ ]:
terms = ['T', 'AP', 'RH', 'V', 'I(T ** 2)', 'I(AP ** 2)', 'I(RH ** 2)', 'I(V ** 2)',
         'T:AP', 'T:RH', 'T:V', 'AP:RH', 'AP:V', 'RH:V']

def interactions_of(effect, terms):
    return [t for t in terms if ':' in t and effect in t.split(':')]

while True:
    m = smf.ols('EP ~ ' + ' + '.join(terms), data=train).fit()
    pvals = m.pvalues.drop('Intercept')
    removable = {}
    for t in terms:
        if ':' not in t and not t.startswith('I(') and interactions_of(t, terms):
            continue  # keep main effect while its interaction is still in the model
        removable[t] = pvals[t]
    worst = max(removable, key=removable.get)
    if removable[worst] < 0.05:
        break
    terms.remove(worst)

mB = smf.ols('EP ~ ' + ' + '.join(terms), data=train).fit()
mseB_train = mean_squared_error(train['EP'], mB.predict(train))
mseB_test = mean_squared_error(test['EP'], mB.predict(test))
print('kept terms:', terms)
print('Model B train MSE=%.3f test MSE=%.3f' % (mseB_train, mseB_test))

Model B's backward elimination keeps T, AP, RH, V, their quadratic terms (except V²), and the T:RH, T:V, AP:RH and AP:V interactions. Its test MSE (about 18.2) is noticeably lower than Model A's (about 20.8), so adding the significant nonlinear and interaction terms does improve prediction, not just training fit. (Which exact terms survive elimination can shift slightly with the random train/test split.)

## (i) KNN regression
Using the same train/test split, fit KNN regression for k = 1 to 100 on both the raw features and the standardized (normalized) features.

In [ ]:
Xtr, Xte = train[predictors].values, test[predictors].values
ytr, yte = train['EP'].values, test['EP'].values

scaler = StandardScaler().fit(Xtr)
Xtr_n, Xte_n = scaler.transform(Xtr), scaler.transform(Xte)

ks = range(1, 101)
raw_train_mse, raw_test_mse, norm_train_mse, norm_test_mse = [], [], [], []

for k in ks:
    knn = KNeighborsRegressor(n_neighbors=k).fit(Xtr, ytr)
    raw_train_mse.append(mean_squared_error(ytr, knn.predict(Xtr)))
    raw_test_mse.append(mean_squared_error(yte, knn.predict(Xte)))

    knn_n = KNeighborsRegressor(n_neighbors=k).fit(Xtr_n, ytr)
    norm_train_mse.append(mean_squared_error(ytr, knn_n.predict(Xtr_n)))
    norm_test_mse.append(mean_squared_error(yte, knn_n.predict(Xte_n)))

inv_k = [1 / k for k in ks]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(inv_k, raw_train_mse, label='train')
axes[0].plot(inv_k, raw_test_mse, label='test')
axes[0].set_title('Raw features')
axes[0].set_xlabel('1/k')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(inv_k, norm_train_mse, label='train')
axes[1].plot(inv_k, norm_test_mse, label='test')
axes[1].set_title('Normalized features')
axes[1].set_xlabel('1/k')
axes[1].set_ylabel('MSE')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
best_k_raw = list(ks)[int(np.argmin(raw_test_mse))]
best_k_norm = list(ks)[int(np.argmin(norm_test_mse))]
print('best k, raw features:', best_k_raw, ' test MSE=%.3f' % min(raw_test_mse))
print('best k, normalized features:', best_k_norm, ' test MSE=%.3f' % min(norm_test_mse))

Normalizing the features helps: the best test MSE with raw features is about 15.7 (at k=5), vs. about 14.1 (at k=4) with normalized features. This makes sense because AP is on a much larger numeric scale (around 1000) than T, RH or V, so raw Euclidean distance is dominated by AP unless the features are put on the same scale first.

## (j) KNN vs. linear regression
Comparing the best test MSE from each approach on the same test set:

| Model | Test MSE |
|---|---|
| Model A: linear, all predictors | ~20.8 |
| Model B: interactions + quadratic terms, backward elimination | ~18.2 |
| KNN, normalized features, best k | ~14.1 |

KNN with normalized features gives the lowest test MSE of the three. This makes sense: KNN does not assume any fixed functional form, so it can automatically pick up the nonlinearities and interactions between T, AP, RH and V that we had to add by hand (and select with p-values) in Model B. The tradeoff is that KNN is less interpretable — it does not give coefficients or p-values that explain how each predictor affects EP, unlike the regression models above.